In [ ]:
# ===================== IMPORTS =====================
import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet161
from sklearn.model_selection import StratifiedKFold
import keras_tuner as kt
import matplotlib.pyplot as plt

# ===================== CONFIG =====================
DATASET_DIR = "/kaggle/input/mangoleafds224x224/MangoLeafDS224x224"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
N_SPLITS = 5
NUM_CLASSES = 6
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ===================== LOAD IMAGE PATHS =====================
image_paths, labels = [], []
class_names = sorted(os.listdir(DATASET_DIR))

for idx, cls in enumerate(class_names):
    cls_dir = os.path.join(DATASET_DIR, cls)
    for f in os.listdir(cls_dir):
        image_paths.append(os.path.join(cls_dir, f))
        labels.append(idx)

image_paths = np.array(image_paths)
labels = np.array(labels)

# ===================== DATA GENERATOR =====================
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i + BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                imgs.append(img)
                labs.append(labels[j])

            imgs = np.array(imgs) / 255.0
            labs = tf.keras.utils.to_categorical(labs, NUM_CLASSES)
            yield imgs, labs

# ===================== MODEL BUILDER =====================
def model_builder(hp):
    base = DenseNet161(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = base.output
    x = layers.GlobalAveragePooling2D()(x)

    dense_units = hp.Choice("dense_units", [512, 1024, 2048])
    dropout_rate = hp.Float("dropout", 0.3, 0.6, step=0.1)

    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs=base.input, outputs=outputs)

    learning_rate = hp.Choice("learning_rate", [1e-3, 1e-4, 1e-5])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# ===================== 5-FOLD CROSS VALIDATION =====================
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels), 1):
    print(f"\n========== FOLD {fold}/{N_SPLITS} ==========")

    tuner = kt.BayesianOptimization(
        model_builder,
        objective="val_accuracy",
        max_trials=10,
        directory="DenseNet161_BayesCV",
        project_name=f"fold_{fold}"
    )

    train_gen = data_generator(image_paths[train_idx], labels[train_idx])
    val_gen   = data_generator(image_paths[val_idx], labels[val_idx])

    tuner.search(
        train_gen,
        steps_per_epoch=len(train_idx) // BATCH_SIZE,
        validation_data=val_gen,
        validation_steps=len(val_idx) // BATCH_SIZE,
        epochs=10,
        verbose=1
    )

    best_model = tuner.get_best_models(num_models=1)[0]

    val_loss, val_acc = best_model.evaluate(
        val_gen,
        steps=len(val_idx) // BATCH_SIZE,
        verbose=0
    )

    print(f"Fold {fold} Validation Accuracy: {val_acc:.4f}")
    fold_accuracies.append(val_acc)

# ===================== FINAL RESULTS =====================
mean_acc = np.mean(fold_accuracies)
std_acc  = np.std(fold_accuracies)

print("\n========== CROSS-VALIDATION SUMMARY ==========")
for i, acc in enumerate(fold_accuracies, 1):
    print(f"Fold {i}: {acc:.4f}")

print(f"\nMean Accuracy: {mean_acc:.4f}")
print(f"Std  Accuracy: {std_acc:.4f}")

# ===================== PLOT =====================
plt.figure(figsize=(7, 4))
plt.plot(range(1, N_SPLITS + 1), fold_accuracies, marker='o')
plt.xlabel("Fold")
plt.ylabel("Validation Accuracy")
plt.title("DenseNet-161 | 5-Fold CV with Bayesian Optimization")
plt.grid(True)
plt.show()
